# Team Challenge · Sprint 03-04 - Catálogo de películas

**Dataset:**  
- [movies.csv](./data/movies.csv)
- [ratings.csv](./data/ratings.csv)
- [tags.csv](./data/tags.csv)
- [links.csv](./data/links.csv) 

**Entregables:** Repositorio de GitHub con el código fuente. Puede ser scripts de python o notebook ordenado, código reproducible, README con cómo ejecutar el proyecto.

**Control de versiones:** gestión del proyecto con **GitHub desde el primer día**. Trabajar en equipo en paralelo con **ramas**, **pull requests** y revisión entre compañeros.

**Parte 1 (Pandas):** Data analytics (Pandas)

**Parte 2 (TMDB):** una petición `GET /3/movie/{id}` por película; definid `TMDB_API_KEY` o `TMDB_READ_ACCESS_TOKEN` en el entorno (sin subir claves al repo).

**Parte 3 (Gemini):** añadir `overview_es` a `movies10` a partir de los `overview` de TMDB; definid `GEMINI_API_KEY` o usad `getpass` (sin subir claves al repo). Requiere `pip install google-genai`.

**Parte 4 (opcional):**
- **A)** catálogo ampliado (`pitch_es`, `edad_sugerida`, `temas`).
- **B)** recomendador por género.

## Trabajo en equipo y control de versiones

- Crear el **repositorio en GitHub** al inicio (día 1–2), no al final.
- Definir una rama principal (`main`) y una para desarrollo (`develop`) y ramas por tarea (p. ej. `feature/parte1-pandas`, `feature/tmdb`, `feature/gemini`).
- Integrar el trabajo con **pull requests** hacia `develop`; al menos **una PR revisada y mergeada por miembro** del equipo.
- Evitar trabajar en en `main` es la rama de "producción"
- Resolver conflictos en ramas antes del merge.
- **README:** cómo clonar el repo, instalar dependencias, configurar claves (`TMDB_*`, `GEMINI_API_KEY`) y ejecutar el notebook o scripts.
- **No subir claves** al repositorio (usar `.gitignore` para `.env` si aplica).

## Parte 1: Data analytics (Pandas)

### 1. Ingesta de datos

- Los CSV están en **`Team_Challenges/TC_01_Sprint_03_04/data/`**.
- Cargar con **Pandas** los cuatro ficheros: `movies.csv`, `ratings.csv`, `tags.csv`, `links.csv`.
- Comprobar para cada `DataFrame`: `shape`, columnas, `dtypes`, `head` y conteo de nulos.

In [3]:
# --- Apartado 1: Ingesta de datos ---
import pandas as pd

# 1. Ruta directa a la carpeta data
DATA_PATH = "data/"

# 2. Carga de los 4 CSV
movies = pd.read_csv(f"{DATA_PATH}movies.csv")
ratings = pd.read_csv(f"{DATA_PATH}ratings.csv")
tags = pd.read_csv(f"{DATA_PATH}tags.csv")
links = pd.read_csv(f"{DATA_PATH}links.csv")

# Diccionario de comprobación
dfs = {
    "movies": movies,
    "ratings": ratings,
    "tags": tags,
    "links": links
}

# 3. Comprobaciones requeridas
for name, df in dfs.items():
    print("=" * 60)
    print(f"DATAFRAME: {name.upper()}")
    print("=" * 60)
    print(f"Dimensiones (shape): {df.shape}")
    print(f"Columnas: {list(df.columns)}")
    print("\nTipos de datos (dtypes):")
    print(df.dtypes)
    print("\nConteo de nulos:")
    print(df.isnull().sum())
    print("\nPrimeras filas:")
    display(df.head())
    print("\n")

DATAFRAME: MOVIES
Dimensiones (shape): (9742, 3)
Columnas: ['movieId', 'title', 'genres']

Tipos de datos (dtypes):
movieId    int64
title        str
genres       str
dtype: object

Conteo de nulos:
movieId    0
title      0
genres     0
dtype: int64

Primeras filas:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy




DATAFRAME: RATINGS
Dimensiones (shape): (100836, 4)
Columnas: ['userId', 'movieId', 'rating', 'timestamp']

Tipos de datos (dtypes):
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object

Conteo de nulos:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Primeras filas:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931




DATAFRAME: TAGS
Dimensiones (shape): (3683, 4)
Columnas: ['userId', 'movieId', 'tag', 'timestamp']

Tipos de datos (dtypes):
userId       int64
movieId      int64
tag            str
timestamp    int64
dtype: object

Conteo de nulos:
userId       0
movieId      0
tag          0
timestamp    0
dtype: int64

Primeras filas:


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200




DATAFRAME: LINKS
Dimensiones (shape): (9742, 3)
Columnas: ['movieId', 'imdbId', 'tmdbId']

Tipos de datos (dtypes):
movieId      int64
imdbId       int64
tmdbId     float64
dtype: object

Conteo de nulos:
movieId    0
imdbId     0
tmdbId     8
dtype: int64

Primeras filas:


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


### 2. Columna `year` desde el título

- En el dataset de películas. el año de estreno suele aparecer **entre paréntesis al final** de `title`, p. ej. `Batman (1989)`.
- Implementad una función (`year_from_title`) que devuelva un entero de cuatro cifras o valor ausente (`NaN` / `<NA>`) si el título no sigue ese patrón.
- Añadid la columna **`year`** a `movies` como numérico (`pd.to_numeric(..., errors="coerce")`).
- Editar el campo `title` para que no contenga el año de estreno.
- Contad cuántas películas **no** tienen año reconocible y mostrad **una pequeña muestra** de sus títulos (casos límite).

In [4]:
# --- Apartado 2: columna 'year' ---
import re
import numpy as np
import pandas as pd

# 1. Función para extraer el año entre paréntesis
def year_from_title(title):
    if not isinstance(title, str):
        return np.nan
    # Busca 4 dígitos entre paréntesis al final del título
    match = re.search(r'\((\d{4})\)\s*$', title.strip())
    if match:
        return int(match.group(1))
    return np.nan

# 2. Crear la columna year numérica
movies['year'] = movies['title'].apply(year_from_title)
movies['year'] = pd.to_numeric(movies['year'], errors="coerce")

# 3. Limpiar el título quitando el año final
movies['title'] = movies['title'].str.replace(r'\s*\(\d{4}\)\s*$', '', regex=True).str.strip()

# 4. Contar y mostrar casos límite (películas sin año reconocible)
sin_year = movies[movies['year'].isna()]
print(f"Películas sin año reconocible: {len(sin_year)}")

print("\nMuestra de casos límite (sin año):")
display(sin_year[['title', 'genres']].head(5))

print("\nMuestra de películas con año extraído correctamente:")
display(movies[['title', 'year', 'genres']].head(5))

Películas sin año reconocible: 13

Muestra de casos límite (sin año):


,title,genres
6059,Babylon 5,Sci-Fi
9031,Ready Player One,Action|Sci-Fi|Thriller
9091,Hyena Road,(no genres listed)
9138,The Adventures of Sherlock Holmes and Doctor W...,(no genres listed)
9179,Nocturnal Animals,Drama|Thriller



Muestra de películas con año extraído correctamente:


,title,year,genres
0,Toy Story,1995.0,Adventure|Animation|Children|Comedy|Fantasy
1,Jumanji,1995.0,Adventure|Children|Fantasy
2,Grumpier Old Men,1995.0,Comedy|Romance
3,Waiting to Exhale,1995.0,Comedy|Drama|Romance
4,Father of the Bride Part II,1995.0,Comedy


### 2. Unificación de datos (*merge* / *join*)

- Entender las **claves** entre tablas (p. ej. `movieId` une `movies`, `ratings`, `tags` y `links`).
- Construir un esquema unificado: al menos una tabla **película–usuario–rating** (ratings enriquecida con título y géneros) y otra con **película–tags** si procede.
- Usar `merge` (u operaciones equivalentes) con criterio claro: tipo de unión (*inner* / *left*), duplicados generados y cómo se resuelven.
- Imprimir las 10 primeras filas de las tablas resultantes.
- Dejar documentado qué filas se pierden o se multiplican al unir y **por qué** es aceptable en vuestro caso de uso.

In [5]:

ratings_movies = pd.merge(
    ratings,
    movies[['movieId', 'title', 'genres', 'year']],
    on='movieId',
    how='inner'
)

# 2. Unificación opcional: película-tags
tags_movies = pd.merge(
    tags,
    movies[['movieId', 'title', 'genres', 'year']],
    on='movieId',
    how='inner'
)

# 3. Documentación del impacto del cruce (filas perdidas o generadas)
print("=== DOCUMENTACIÓN DE UNIONES ===")
print(f"Filas originales en ratings: {len(ratings)}")
print(f"Filas resultantes en ratings_movies: {len(ratings_movies)}")
print(f"Diferencia de filas: {len(ratings) - len(ratings_movies)}")

print(f"\nPelículas únicas en movies: {movies['movieId'].nunique()}")
print(f"Películas únicas con rating: {ratings_movies['movieId'].nunique()}")
print(f"Películas del catálogo sin valoraciones: {movies['movieId'].nunique() - ratings_movies['movieId'].nunique()}")

# 4. Mostrar las 10 primeras filas de la tabla resultante
print("\nPrimeras 10 filas de ratings_movies:")
display(ratings_movies.head(10))

=== DOCUMENTACIÓN DE UNIONES ===
Filas originales en ratings: 100836
Filas resultantes en ratings_movies: 100836
Diferencia de filas: 0

Películas únicas en movies: 9742
Películas únicas con rating: 9724
Películas del catálogo sin valoraciones: 18

Primeras 10 filas de ratings_movies:


,userId,movieId,rating,timestamp,title,genres,year
0,1,1,4.0,964982703,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995.0
1,1,3,4.0,964981247,Grumpier Old Men,Comedy|Romance,1995.0
2,1,6,4.0,964982224,Heat,Action|Crime|Thriller,1995.0
3,1,47,5.0,964983815,Seven (a.k.a. Se7en),Mystery|Thriller,1995.0
4,1,50,5.0,964982931,"Usual Suspects, The",Crime|Mystery|Thriller,1995.0
5,1,70,3.0,964982400,From Dusk Till Dawn,Action|Comedy|Horror|Thriller,1996.0
6,1,101,5.0,964980868,Bottle Rocket,Adventure|Comedy|Crime|Romance,1996.0
7,1,110,4.0,964982176,Braveheart,Action|Drama|War,1995.0
8,1,151,5.0,964984041,Rob Roy,Action|Drama|Romance|War,1995.0
9,1,157,5.0,964984100,Canadian Bacon,Comedy|War,1995.0


### 4. Agregaciones y segmentación

- Usar `groupby` (por usuario, por película o por género) con al menos una agregación multi-columna (`agg`).
- Comparar dos segmentos (p. ej. usuarios con muchas valoraciones vs. pocos; o por **década** usando la columna `year` del apartado 3) con tablas resumen.

In [6]:
# --- Apartado 4: Agregaciones y segmentación ---

# 1. Agregación multi-columna: métricas por película
movie_stats = ratings_movies.groupby(['movieId', 'title', 'year']).agg(
    num_ratings=('rating', 'count'),
    mean_rating=('rating', 'mean'),
    std_rating=('rating', 'std')
).reset_index()

print("Top 5 películas con más valoraciones:")
display(movie_stats.sort_values(by='num_ratings', ascending=False).head(5))

# 2. Segmentación A: Usuarios muy activos vs. poco activos
# Definimos el umbral en la mediana de valoraciones por usuario
user_counts = ratings_movies.groupby('userId')['rating'].count()
median_activity = user_counts.median()

ratings_movies['segmento_usuario'] = ratings_movies['userId'].apply(
    lambda x: 'Usuario Muy Activo' if user_counts[x] > median_activity else 'Usuario Poco Activo'
)

segmento_usuarios_summary = ratings_movies.groupby('segmento_usuario').agg(
    total_usuarios=('userId', 'nunique'),
    total_ratings=('rating', 'count'),
    media_rating=('rating', 'mean'),
    desv_rating=('rating', 'std')
).reset_index()

print("\nComparación de segmentos: Usuarios Activos vs. Poco Activos:")
display(segmento_usuarios_summary)

# 3. Segmentación B: Análisis y comparación por década (usando 'year')
ratings_movies['decade'] = (ratings_movies['year'] // 10) * 10

decade_summary = ratings_movies.dropna(subset=['decade']).groupby('decade').agg(
    total_peliculas=('movieId', 'nunique'),
    total_valoraciones=('rating', 'count'),
    media_rating=('rating', 'mean')
).reset_index().sort_values(by='decade', ascending=False)

# Formatear década como entero (ej. 1990 en vez de 1990.0)
decade_summary['decade'] = decade_summary['decade'].astype(int).astype(str) + 's'

print("\nComparación de métricas por Década:")
display(decade_summary.head(8))

Top 5 películas con más valoraciones:


,movieId,title,year,num_ratings,mean_rating,std_rating
314,356,Forrest Gump,1994.0,329,4.164134,0.831244
277,318,"Shawshank Redemption, The",1994.0,317,4.429022,0.713019
257,296,Pulp Fiction,1994.0,307,4.197068,0.951997
510,593,"Silence of the Lambs, The",1991.0,279,4.161290,0.853983
1938,2571,"Matrix, The",1999.0,278,4.192446,0.975243



Comparación de segmentos: Usuarios Activos vs. Poco Activos:


,segmento_usuario,total_usuarios,total_ratings,media_rating,desv_rating
0,Usuario Muy Activo,305,89129,3.473303,1.036398
1,Usuario Poco Activo,305,11707,3.716665,1.063794



Comparación de métricas por Década:


,decade,total_peliculas,total_valoraciones,media_rating
11,2010s,1930,9487,3.488774
10,2000s,2848,29766,3.465750
9,1990s,2209,37087,3.434613
8,1980s,1175,12912,3.518355
7,1970s,498,4995,3.775676
6,1960s,399,2858,3.808083
5,1950s,277,1784,3.845011
4,1940s,194,1101,3.870572


### 5. Preguntas sobre los datos (`movies`)

**Requisito:** haber creado la columna `year` en el **apartado 2**.

1. **Cuántas** películas están listadas en `movies`.
2. **Cuáles** son las **más antiguas** (menor año extraído del título).
3. **Cuántas** tienen **"Dracula"** en el título (coincidencia parcial, sin distinguir mayúsculas).
4. **Títulos más comunes**.
5. Películas con **"Exorcist"** ordenadas de la más antigua a la más moderna.
6. **Cuántas** con año **1950**.
7. **Cuántas** entre **1950 y 1959** inclusive.
8. **Año** de la película con título exacto **`Batman`** (y contraste con otras *Batman*).
9. Listado de películas que tienen como tag "sci-fi" y "adventure"
10. ¿Cuál es la tag más repetida?

In [7]:
# --- Apartado 5: Preguntas sobre `movies` ---

print("=" * 60)
print("RESPUESTAS A LAS 10 PREGUNTAS SOBRE LOS DATOS")
print("=" * 60)

# 1. Cuántas películas están listadas en movies
total_movies = len(movies)
print(f"\n1. Películas listadas en 'movies': {total_movies}")

# 2. Cuáles son las más antiguas (menor año extraído)
min_year = movies['year'].min()
oldest_movies = movies[movies['year'] == min_year][['title', 'year', 'genres']]
print(f"\n2. Películas más antiguas (Año {int(min_year)}):")
display(oldest_movies)

# 3. Cuántas tienen 'Dracula' en el título (case-insensitive)
dracula_count = movies['title'].str.contains('Dracula', case=False, na=False).sum()
print(f"\n3. Películas con 'Dracula' en el título: {dracula_count}")

# 4. Títulos más comunes
title_counts = movies['title'].value_counts()
print(f"\n4. Títulos más repetidos en el catálogo:")
display(title_counts[title_counts > 1].head(5))

# 5. Películas con 'Exorcist' ordenadas de la más antigua a la más moderna
exorcist_movies = movies[movies['title'].str.contains('Exorcist', case=False, na=False)].sort_values(by='year')
print(f"\n5. Películas con 'Exorcist' ordenadas por año:")
display(exorcist_movies[['title', 'year', 'genres']])

# 6. Cuántas con año 1950
movies_1950 = (movies['year'] == 1950).sum()
print(f"\n6. Películas estrenadas exactamente en 1950: {movies_1950}")

# 7. Cuántas entre 1950 y 1959 inclusive
movies_50s = movies['year'].between(1950, 1959).sum()
print(f"\n7. Películas estrenadas entre 1950 y 1959 (inclusive): {movies_50s}")

# 8. Año con título exacto 'Batman' y contraste con otras películas de Batman
batman_exact = movies[movies['title'].str.strip() == 'Batman']
batman_all = movies[movies['title'].str.contains('Batman', case=False, na=False)]
print(f"\n8. Año de la película con título exacto 'Batman': {int(batman_exact['year'].values[0]) if not batman_exact.empty else 'No encontrada'}")
print("   Contraste con todas las películas que contienen 'Batman':")
display(batman_all[['title', 'year', 'genres']])

# 9. Películas que tienen como tag 'sci-fi' y 'adventure'
# Normalizamos tags a minúsculas para asegurar coincidencias limpias
tags_norm = tags.copy()
tags_norm['tag_clean'] = tags_norm['tag'].astype(str).str.strip().str.lower()

scifi_movies = tags_norm[tags_norm['tag_clean'] == 'sci-fi']['movieId'].unique()
adventure_movies = tags_norm[tags_norm['tag_clean'] == 'adventure']['movieId'].unique()
both_tags_ids = set(scifi_movies).intersection(set(adventure_movies))

movies_scifi_adv = movies[movies['movieId'].isin(both_tags_ids)]
print(f"\n9. Películas con ambos tags ('sci-fi' y 'adventure') [{len(movies_scifi_adv)} encontradas]:")
display(movies_scifi_adv[['title', 'year', 'genres']])

# 10. Tag más repetida
top_tag = tags_norm['tag_clean'].value_counts().head(1)
print(f"\n10. Tag más repetida en el dataset:")
print(f"    Tag: '{top_tag.index[0]}' con {top_tag.values[0]} apariciones.")

RESPUESTAS A LAS 10 PREGUNTAS SOBRE LOS DATOS

1. Películas listadas en 'movies': 9742

2. Películas más antiguas (Año 1902):


,title,year,genres
5868,"Trip to the Moon, A (Voyage dans la lune, Le)",1902.0,Action|Adventure|Fantasy|Sci-Fi



3. Películas con 'Dracula' en el título: 9

4. Títulos más repetidos en el catálogo:


title
Hamlet                   5
Misérables, Les          4
Three Musketeers, The    4
Jane Eyre                4
Christmas Carol, A       4
Name: count, dtype: int64


5. Películas con 'Exorcist' ordenadas por año:


...


6. Películas estrenadas exactamente en 1950: 21

7. Películas estrenadas entre 1950 y 1959 (inclusive): 279

8. Año de la película con título exacto 'Batman': 1989
   Contraste con todas las películas que contienen 'Batman':


,title,year,genres
126,Batman Forever,1995.0,Action|Adventure|Comedy|Crime
509,Batman,1989.0,Action|Crime|Thriller
1060,Batman Returns,1992.0,Action|Crime
1174,Batman & Robin,1997.0,Action|Adventure|Fantasy|Thriller
2418,Batman: Mask of the Phantasm,1993.0,Animation|Children
5463,Batman,1966.0,Action|Adventure|Comedy
5620,"Batman/Superman Movie, The",1998.0,Action|Adventure|Animation|Children|Fantasy|Sc...
5631,Batman Beyond: Return of the Joker,2000.0,Action|Animation|Crime|Sci-Fi|Thriller
5917,Batman Begins,2005.0,Action|Crime|IMAX
6815,Batman: Gotham Knight,2008.0,Action|Animation|Crime



9. Películas con ambos tags ('sci-fi' y 'adventure') [0 encontradas]:


,title,year,genres



10. Tag más repetida en el dataset:
    Tag: 'in netflix queue' con 131 apariciones.


## Parte 2: Petición HTTP a la API de TMDB

### Endpoint

`GET https://api.themoviedb.org/3/movie/{tmdb_id}`

Ejemplo público de la misma forma que en la documentación de TMDB: **`https://api.themoviedb.org/3/movie/100`** (el número es el `tmdb_id`; en vuestro caso usaréis el `tmdbId` de cada fila de `links`).

### Tareas

1. Construir un **`DataFrame` `movies10`** con **10 películas** del dataset original (por ejemplo las 10 primeras filas que tengan `tmdbId` tras unir `movies` con `links`).
2. Escribir una función **`fetch_movie_details(tmdb_id)`** que haga la petición anterior y devuelva al menos **`overview`** y **`homepage`** (texto vacío si vienen nulos).
3. Recorrer `movies10` y **añadir** a cada registro esas dos columnas en el propio `DataFrame`.

### Autenticación

Para poder acceder a la API de TMDB, debéis hacer uso del ACCESS TOKEN o de la API KEY que te proporcionan al registrarse. Recomendamos probar los endpoint en POSTMAN para hacer las pruebas de la llamada a la API antes de crear el script en Python. Deberíais tener en vuestro proyecto:**`TMDB_API_KEY`** (query `api_key`) o **`TMDB_READ_ACCESS_TOKEN`** (cabecera `Authorization: Bearer …`). Podéis crear un proyecto con variables de entorno o usar una celda de código usando **`getpass`** (entrada oculta, solo esa sesión del kernel). 

No guardéis claves en el notebook

In [ ]:
# --- Parte 2: TMDB GET /movie/{id} → overview y homepage  ---
# Requiere: `movies` y `links` cargados. pip install requests

## Parte 3: Sinopsis en español con Gemini

Usad el `DataFrame` **`movies10`** de la Parte 2 (columna `overview` en inglés desde TMDB).

### Tareas

1. Configurar **`GEMINI_API_KEY`** (variable de entorno o `getpass`, como en Sprint 4).
2. Crear el cliente **`google-genai`** y una función **`summarize_overview_es(overview, title="")`** que devuelva un **resumen en español de máximo 2 frases**. Si `overview` está vacío, devolver cadena vacía **sin llamar a la API**.
3. Recorrer `movies10` y añadir la columna **`overview_es`**.
4. Mostrar `title`, `overview` (recorte) y `overview_es` para **3 películas**.

### Autenticación

Clave en [Google AI Studio](https://aistudio.google.com/). Variable **`GEMINI_API_KEY`** o celda con **`getpass`**. No guardéis claves en el notebook.

### Prerrequisitos

- Parte 2 ejecutada (`movies10` con columna `overview`).
- `pip install google-genai`

In [ ]:
# --- Parte 3: overview → overview_es con Gemini  ---
# Requiere: `movies10` de la Parte 2. pip install google-genai

In [ ]:
MODEL = 'gemini-3.1-flash-lite'

In [ ]:
from google import genai

client = genai.Client()

interaction = client.interactions.create(
    model="gemini-3.6-flash",
    input="Explain how AI works in a few words"
)
print(interaction.output_text)

## Parte 4 (opcional): extensiones con Gemini

**No obligatorio.** Solo si el grupo terminó las partes 1-3. Podéis elegir **A**, **B** o ambas. Son independientes.

---

### A) Catálogo ampliado en `movies10`

Partiendo de `movies10` (con `overview` y, si ya lo tenéis, `overview_es`), añadid con **una llamada JSON por película**:

- **`pitch_es`**: texto de cartelera en español (máx. 280 caracteres).
- **`edad_sugerida`**: uno de `TP`, `+7`, `+12`, `+16`, `+18`.
- **`temas`**: exactamente 3 temas en español (en el DataFrame, cadena separada por comas).

Función sugerida: **`enrich_catalog_fields(overview, title="", genres="", year=None)`**. Basad la respuesta solo en sinopsis y metadatos; no inventéis reparto ni datos externos. Si `overview` está vacío, no llaméis a la API.

Mostrad 2–3 filas con las columnas nuevas.

---

### B) Recomendador por género

Usad **`ratings_movies`** (Parte 1) y Gemini.

1. **`top_rated_by_genre(ratings_movies, genres, top_n=10, min_ratings=50)`** — filtra por género(s), nota media por `movieId`, devuelve el top N (mínimo `min_ratings` valoraciones por película).
2. **`recommend_movies(candidates, favorite_genres, n=3)`** — el modelo recomienda **solo** del catálogo candidato, con una frase de justificación en español cada una.
3. Probad con 1–2 géneros (p. ej. `["Action", "Sci-Fi"]`): mostrad candidatas y respuesta del modelo.

In [ ]:
# --- Parte 4A (opcional): pitch_es, edad_sugerida, temas ---
# Requiere: `movies10` con `overview`; `client` y `MODEL` de la Parte 3.

In [ ]:
# --- Parte 4B (opcional): recomendador por género (SOLUCIÓN) ---
# Requiere: `ratings_movies` (Parte 1); `client` y `MODEL` (Parte 3).